Gold overview

## Gold Layer (Business Metrics for BI)

**Goal:** Build reusable churn KPIs and segment-wise churn metrics.  
Gold layer is what dashboards and stakeholders consume.


Create Gold Schema

In [0]:
CREATE SCHEMA IF NOT EXISTS workspace.gold;


Base view  

Creates a reusable “base” view from silver.
All metric views will use this so logic stays consistent.
Prevents repeating the same SELECT in every metric view.

In [0]:
CREATE OR REPLACE VIEW workspace.gold.vw_churn_base AS
SELECT
  customerID,
  gender,
  senior_citizen_flag,
  Partner,
  Dependents,
  tenure,
  tenure_bucket,
  Contract,
  PaymentMethod,
  MonthlyCharges,
  TotalCharges,
  churn_flag
FROM silver.telco_churn_customers;


In [0]:
SELECT * FROM workspace.gold.vw_churn_base LIMIT 10;


customerID,gender,senior_citizen_flag,Partner,Dependents,tenure,tenure_bucket,Contract,PaymentMethod,MonthlyCharges,TotalCharges,churn_flag
7590-VHVEG,Female,0,Yes,No,1,0–12 months,Month-to-month,Electronic check,29.85,29.85,0
5575-GNVDE,Male,0,No,No,34,25–48 months,One year,Mailed check,56.95,1889.5,0
3668-QPYBK,Male,0,No,No,2,0–12 months,Month-to-month,Mailed check,53.85,108.15,1
7795-CFOCW,Male,0,No,No,45,25–48 months,One year,Bank transfer (automatic),42.3,1840.75,0
9237-HQITU,Female,0,No,No,2,0–12 months,Month-to-month,Electronic check,70.7,151.65,1
9305-CDSKC,Female,0,No,No,8,0–12 months,Month-to-month,Electronic check,99.65,820.5,1
1452-KIOVK,Male,0,No,Yes,22,13–24 months,Month-to-month,Credit card (automatic),89.1,1949.4,0
6713-OKOMC,Female,0,No,No,10,0–12 months,Month-to-month,Mailed check,29.75,301.9,0
7892-POOKP,Female,0,Yes,No,28,25–48 months,Month-to-month,Electronic check,104.8,3046.05,1
6388-TABGU,Male,0,No,Yes,62,49+ months,One year,Bank transfer (automatic),56.15,3487.95,0


Overall KPIs

Calculates total customers, churned customers, retained customers.
Calculates churn rate and retention rate.
Uses * 1.0 to avoid integer division (important).

In [0]:
CREATE OR REPLACE VIEW workspace.gold.vw_churn_kpis_overall AS
SELECT
  COUNT(*)                                  AS total_customers,
  SUM(churn_flag)                           AS churned_customers,
  COUNT(*) - SUM(churn_flag)                AS retained_customers,
  ROUND(SUM(churn_flag) * 1.0 / COUNT(*), 4)              AS churn_rate,
  ROUND((COUNT(*) - SUM(churn_flag)) * 1.0 / COUNT(*), 4) AS retention_rate
FROM gold.vw_churn_base;


In [0]:
SELECT * FROM workspace.gold.vw_churn_kpis_overall;


total_customers,churned_customers,retained_customers,churn_rate,retention_rate
7043,1869,5174,0.2654,0.7346


Churn by contract

Groups customers by contract type (month-to-month / one year / two year).
Calculates churn rate for each contract segment.
Sorts segments by highest churn first.

In [0]:
CREATE OR REPLACE VIEW workspace.gold.vw_churn_by_contract AS
SELECT
  Contract,
  COUNT(*) AS total_customers,
  SUM(churn_flag) AS churned_customers,
  ROUND(SUM(churn_flag) * 1.0 / COUNT(*), 4) AS churn_rate
FROM gold.vw_churn_base
GROUP BY Contract
ORDER BY churn_rate DESC;


In [0]:
SELECT * FROM gold.vw_churn_by_contract;


Contract,total_customers,churned_customers,churn_rate
Month-to-month,3875,1655,0.4271
One year,1473,166,0.1127
Two year,1695,48,0.0283


Churn by tenure bucket

Shows how churn changes across customer lifecycle buckets.
Helps identify if churn is early-stage or late-stage.

In [0]:
CREATE OR REPLACE VIEW workspace.gold.vw_churn_by_tenure_bucket AS
SELECT
  tenure_bucket,
  COUNT(*) AS total_customers,
  SUM(churn_flag) AS churned_customers,
  ROUND(SUM(churn_flag) * 1.0 / COUNT(*), 4) AS churn_rate
FROM gold.vw_churn_base
GROUP BY tenure_bucket
ORDER BY churn_rate DESC;


In [0]:
SELECT * FROM gold.vw_churn_by_tenure_bucket;


tenure_bucket,total_customers,churned_customers,churn_rate
0–12 months,2186,1037,0.4744
13–24 months,1024,294,0.2871
25–48 months,1594,325,0.2039
49+ months,2239,213,0.0951


Churn by monthly charge band

Creates pricing bands and computes churn rate per band.
Also provides average monthly charges per band.
Useful to understand churn patterns for low vs high-paying customers.

In [0]:
CREATE OR REPLACE VIEW workspace.gold.vw_churn_by_monthly_charge_band AS
SELECT
  CASE
    WHEN MonthlyCharges < 30 THEN "<30"
    WHEN MonthlyCharges < 60 THEN "30-59"
    WHEN MonthlyCharges < 90 THEN "60-89"
    ELSE "90+"
  END AS monthly_charge_band,
  COUNT(*) AS total_customers,
  SUM(churn_flag) AS churned_customers,
  ROUND(SUM(churn_flag) * 1.0 / COUNT(*), 4) AS churn_rate,
  ROUND(AVG(MonthlyCharges), 2) AS avg_monthly_charges
FROM gold.vw_churn_base
GROUP BY
  CASE
    WHEN MonthlyCharges < 30 THEN "<30"
    WHEN MonthlyCharges < 60 THEN "30-59"
    WHEN MonthlyCharges < 90 THEN "60-89"
    ELSE "90+"
  END
ORDER BY churn_rate DESC;


In [0]:
SELECT * FROM workspace.gold.vw_churn_by_monthly_charge_band;


monthly_charge_band,total_customers,churned_customers,churn_rate,avg_monthly_charges
60-89,2392,807,0.3374,76.72
90+,1744,573,0.3286,100.99
30-59,1254,327,0.2608,48.59
<30,1653,162,0.0980,21.5


Revenue impact proxy

Estimates how much monthly revenue is linked to churned customers.
Uses MonthlyCharges as a proxy since the dataset is not time-series MRR.
NULLIF avoids divide-by-zero errors.

In [0]:
CREATE OR REPLACE VIEW workspace.gold.vw_revenue_impact AS
SELECT
  ROUND(SUM(MonthlyCharges), 2) AS total_monthly_revenue_proxy,
  ROUND(SUM(CASE WHEN churn_flag = 1 THEN MonthlyCharges ELSE 0 END), 2) AS churned_monthly_revenue_proxy,
  ROUND(
    SUM(CASE WHEN churn_flag = 1 THEN MonthlyCharges ELSE 0 END) * 1.0 /
    NULLIF(SUM(MonthlyCharges), 0),
    4
  ) AS revenue_at_risk_share
FROM gold.vw_churn_base;


In [0]:
SELECT * FROM gold.vw_revenue_impact;


total_monthly_revenue_proxy,churned_monthly_revenue_proxy,revenue_at_risk_share
456116.6,139130.85,0.305


In [0]:
SHOW VIEWS IN gold;


namespace,viewName,isTemporary,isMaterialized,isMetric
gold,vw_churn_base,false,false,false
gold,vw_churn_by_contract,false,false,false
gold,vw_churn_by_monthly_charge_band,false,false,false
gold,vw_churn_by_tenure_bucket,false,false,false
gold,vw_churn_kpis_overall,false,false,false
gold,vw_revenue_impact,false,false,false
